# Multi-Stage Minimum Failure Example (MFE) Time Stepping Simulation

This notebook implements a multi-stage Runge-Kutta-like time integration scheme for solving a PDE system using Devito.

## Overview

To apply multi-stage methods to a PDE system, we first have to formulated as a first-order in time system:


$\frac{d}{dt}\boldsymbol{U}(\boldsymbol{x},t)=\boldsymbol{HU}(\boldsymbol{x},t)+\boldsymbol{f}(\boldsymbol{x},t),\hspace{3cm}(1)$

where $\boldsymbol{x}$ is a vector with the spatial variables (x in 1D, (x,y) in 2D, (x,y,z in 3D)), $\boldsymbol{U}$ is a vector of real-valued functions, $\boldsymbol{H}$ is a spatial operator that contains spatial derivatives and constant coefficients, and $\boldsymbol{f}$ is a known vector of real-valued functions.

Expanding $\boldsymbol{f}(x,y,t)$ in its Taylor 's series, we can approximate the solution of system (1) by the matrix exponential (see [[1](#ref-higham2010)])

$\hat{\boldsymbol{U}}(\boldsymbol{x},t)\approx[\boldsymbol{I_p}\;0]e^{t\hat{\boldsymbol{H}}}\begin{bmatrix}\boldsymbol{U}(\boldsymbol{x},t_0)\\ \boldsymbol{e_p}\end{bmatrix},$

where $\boldsymbol{e_p}\in\mathbb{R}^p$ is the eigenvector with zero in all its entries exept the last one, that equals 1, $\hat{\boldsymbol{U}}$ is the vector function $\boldsymbol{U}$ with $p$ extra zeros at the end, $\boldsymbol{I_p}$ is the identity matrix of size $p$, and $\hat{\boldsymbol{H}}$ is the matrix

$\hat{\boldsymbol{H}}=\begin{bmatrix}\boldsymbol{H} & \boldsymbol{W}\\ 0 & \boldsymbol{J_p}\end{bmatrix}, \hspace{5.27cm}(2)$

where $\boldsymbol{J_p}$ is the zero matrix with an upper diagonal of ones

$\boldsymbol{J_p}=\begin{bmatrix}0 & 1 & 0 & 0 & \dots & 0\\ 0 & 0 & 1 & 0 & \dots & 0\\ &&\ddots&&&\vdots 
\\ 0 & 0 & 0 & 0 & \dots & 1 \\0 & 0 & 0 & 0 & \dots & 0\end{bmatrix},$

and $\boldsymbol{W}$ contains the information of the vector function $\boldsymbol{f}(\boldsymbol{x},t)$ derivatives

$\boldsymbol{W}=\begin{bmatrix}\frac{\partial^{p-1}}{\partial t^{p-1}}\boldsymbol{f}(\boldsymbol{x},t_0)\bigg\vert \frac{\partial^{p-2}}{\partial t^{p-2}}\boldsymbol{f}(\boldsymbol{x},t_0)\bigg\vert \dots \bigg\vert \frac{\partial}{\partial t}\boldsymbol{f}(\boldsymbol{x},t_0)\bigg\vert \boldsymbol{f}(\boldsymbol{x},t_0)\end{bmatrix}.$

Then, we approximate the solution operator in (2) with the m-stage Runge-Kutta (RK) method from [[2](#ref-gottlieb2003)]

\begin{align*}
        \boldsymbol{k}_0&=\boldsymbol{u}_n\\
        \boldsymbol{k}_i&=\left(\boldsymbol{I}_{n\times n}+\Delta t \hat{\boldsymbol{H}}\right)\boldsymbol{k}_{i-1},\quad i=1\dots m-1\\
        \boldsymbol{u}_{n+1}&=\sum\limits_{i=0}^{m-2}\alpha_i\boldsymbol{k}_i+\alpha_{m-1}\left(\boldsymbol{I}_{n\times n}+\Delta t \hat{\boldsymbol{H}}\right)\boldsymbol{k}_{m-1},        
\end{align*}

where $\alpha_i$ are the coefficients of the Runge-Kutta and have a straightforward computation.

So, for each time step we have to construct the matrix $\hat{\boldsymbol{H}}$, where we only the submatrix $\boldsymbol{W}$ change, and apply the RK method to the vector $\boldsymbol{u}_n=[\boldsymbol{U}(\boldsymbol{x},t_n)\;\; \boldsymbol{e_p}]^T$.

So, an outline of the implementation is:

- **Environmental variables**: Set up the grid, and spatial and time variables
- **PDE System definition**: Define $\boldsymbol{f}(\boldsymbol{x},t)$ and the system of equations
- **Compute the derivatives of the source term**: symbolic computing of $\boldsymbol{f}(\boldsymbol{x},t)$ derivatives
- **Construct the operator $\hat{\boldsymbol{H}}$**: application the application of operator $\hat{\boldsymbol{H}}$
- **Implementation of the RK method**: define the required equations of the RK method to pass to Devito's operator. For this particular example, we'll use $m=3$ and $\alpha_i=1,\;\forall i\in\{0,1,2\}$.
- **Create and run Devito operator**: Executing the operator constructed with the RK equations

## 0. Import Required Libraries

First, we import all necessary libraries including NumPy for numerical operations, SymPy for symbolic mathematics, and Devito components for finite difference operations.

In [9]:
import numpy as np
import sympy as sym
from devito import (Grid, Function, TimeFunction, VectorTimeFunction,
                    Derivative, Operator, Eq, VectorFunction)
from devito import configuration
from devito.symbolics import uxreplace

## 1. Environmental variables

Configure the simulation environment, including logging level, domain parameters, and computational grid setup.

In [10]:
# Configure Devito logging
configuration['log-level'] = 'DEBUG'

# Simulation parameters
extent = (1, 1)           # Physical domain size
shape = (201, 201)        # Grid resolution
origin = (0, 0)           # Domain origin

# Create computational grid
grid = Grid(origin=origin, extent=extent, shape=shape, dtype=np.float64)
x, y = grid.dimensions
t, dt = grid.time_dim, grid.stepping_dim.spacing

print(f"Grid created: {shape} points over domain {extent}")
print(f"Spatial dimensions: x={x}, y={y}")
print(f"Time dimension: t={t}, dt={dt}")

Grid created: (201, 201) points over domain (1, 1)
Spatial dimensions: x=x, y=y
Time dimension: t=time, dt=dt


## 2. PDE System definition

Create the TimeFunction objects for the wavefield variables (displacement u and velocity v) and define the source terms with both spatial and temporal characteristics, to define the system of equations

\begin{align*}
\frac{d}{dt}\begin{pmatrix}u\\v\end{pmatrix}=\begin{pmatrix}0&1\\\frac{\partial^2}{\partial x^2}+\frac{\partial^2}{\partial y^2}&0\end{pmatrix}\begin{pmatrix}u\\v\end{pmatrix} + \begin{pmatrix}f_1(x,y,t)\\0\end{pmatrix} + \begin{pmatrix}f_2(x,y,t)\\0\end{pmatrix}+\begin{pmatrix}0\\f_3(x,y,t)\end{pmatrix}.
\end{align*}

In [11]:
# Define wavefield unknowns: u (displacement) and v (velocity)
fun_labels = ['vx', 'vy','txx','txy','tyy']
field_defs = [TimeFunction(name=name, grid=grid, space_order=2, time_order=0, dtype=np.float64) for name in fun_labels]
u_multi_stage = VectorTimeFunction(name='u_vec', grid=grid, components=field_defs)

print("Wavefield functions created:")
for i, u in enumerate(field_defs):
    print(f"  {fun_labels[i]}: {u}")

# Source definition
src_spatial = Function(name="src_spat", grid=grid, space_order=2, dtype=np.float64)
src_spatial.data[100, 100] = 1  # Point source at grid center
src_temporal = sym.exp(- 100 * (t - 0.01) ** 2)  # Gaussian pulse

print(f"\nSource spatial function: {src_spatial}")
print(f"Source temporal function: {src_temporal}")

# PDE right-hand side: du/dt = v, dv/dt = ∇²u
system_eqs_rhs = VectorFunction(grid=grid, components=(Derivative(u_multi_stage[2],x,fd_order=2)+Derivative(u_multi_stage[3],y,fd_order=2),
                                                       Derivative(u_multi_stage[3],x,fd_order=2)+Derivative(u_multi_stage[4],y,fd_order=2),
                                                       Derivative(u_multi_stage[0],x,fd_order=2)+Derivative(u_multi_stage[1],y,fd_order=2),
                                                       Derivative(u_multi_stage[0],y,fd_order=2)+Derivative(u_multi_stage[1],x,fd_order=2),
                                                       Derivative(u_multi_stage[0],x,fd_order=2)+Derivative(u_multi_stage[1],y,fd_order=2)))
# Source coupling f1, f2, f3: [spatial, temporal, associated variable]
src = [[src_spatial, src_temporal, u_multi_stage[0]],
       [src_spatial, src_temporal * 10, u_multi_stage[0]],
       [src_spatial, src_temporal, u_multi_stage[1]]]

print(f"\nPDE system matrix H:")
for i, rhs in enumerate(system_eqs_rhs):
    print(f"  d{fun_labels[i]}/dt = {rhs}")


Allocating host memory for src_spat(205, 205) [328 KB]


Wavefield functions created:
  vx: vx(t, x, y)
  vy: vy(t, x, y)
  txx: txx(t, x, y)
  txy: txy(t, x, y)
  tyy: tyy(t, x, y)

Source spatial function: src_spat(x, y)
Source temporal function: exp(-100*(time - 0.01)**2)

PDE system matrix H:
  dvx/dt = Derivative(txx(t, x, y), x) + Derivative(txy(t, x, y), y)
  dvy/dt = Derivative(txy(t, x, y), x) + Derivative(tyy(t, x, y), y)
  dtxx/dt = Derivative(vx(t, x, y), x) + Derivative(vy(t, x, y), y)
  dtxy/dt = Derivative(vx(t, x, y), y) + Derivative(vy(t, x, y), x)
  dtyy/dt = Derivative(vx(t, x, y), x) + Derivative(vy(t, x, y), y)


## 3. Compute the derivatives of the source term

Implement the core helper function that compute time derivatives of source and calculate the source derivatives up to the specified degree (deg=3).

In [12]:
def source_derivatives(deg, src, src_index, t):
    """
    Compute time derivatives of the source up to given degree.
    
    Parameters:
    -----------
    deg : int
        Degree of derivatives to compute
    src : list
        List of source terms
    src_index : list
        Indices for source terms
    t : symbol
        Time symbol
        
    Returns:
    --------
    f_deriv : list
        List of derivative expressions
    """
    f_deriv = [[src[i][1] for i in range(len(src))]]
    
    # Compute derivatives up to order p
    for _ in range(deg - 1):
        f_deriv.append([f_deriv[-1][i].diff(t) for i in range(len(src_index))])
    
    f_deriv.reverse()
    return f_deriv

print("  - source_derivatives(): Computes time derivatives of source wavelet")

# Create mapping from wavefield variables to indices
src_index_map = {val: i for i, val in enumerate(u_multi_stage)}
print("Source index mapping:")
for var, idx in src_index_map.items():
    print(f"  {var} -> {idx}")

# Extract source indices based on associated variables
src_index = [src_index_map[val] for val in [src[i][2] for i in range(len(src))]]
print(f"\nSource indices: {src_index}")

# Degree of derivatives to compute
deg = 3

# Compute source derivatives
src_deriv = source_derivatives(deg, src, src_index, t)
print(f"\nSource derivatives computed up to degree {deg}")
print(f"Number of derivative levels: {len(src_deriv)}")
print("Sample derivative expressions:")
for i, deriv in enumerate(src_deriv):
    print(f"  Level {deg-1-i}: {deriv}")

  - source_derivatives(): Computes time derivatives of source wavelet
Source index mapping:
  vx(t, x, y) -> 0
  vy(t, x, y) -> 1
  txx(t, x, y) -> 2
  txy(t, x, y) -> 3
  tyy(t, x, y) -> 4

Source indices: [0, 0, 1]

Source derivatives computed up to degree 3
Number of derivative levels: 3
Sample derivative expressions:
  Level 2: [(2.0 - 200*time)**2*exp(-100*(time - 0.01)**2) - 200*exp(-100*(time - 0.01)**2), 10*(2.0 - 200*time)**2*exp(-100*(time - 0.01)**2) - 2000*exp(-100*(time - 0.01)**2), (2.0 - 200*time)**2*exp(-100*(time - 0.01)**2) - 200*exp(-100*(time - 0.01)**2)]
  Level 1: [(2.0 - 200*time)*exp(-100*(time - 0.01)**2), 10*(2.0 - 200*time)*exp(-100*(time - 0.01)**2), (2.0 - 200*time)*exp(-100*(time - 0.01)**2)]
  Level 0: [exp(-100*(time - 0.01)**2), 10*exp(-100*(time - 0.01)**2), exp(-100*(time - 0.01)**2)]


## 4.Construct the operator $\hat{\boldsymbol{H}}$

 Application of the spatial operator to the vector formed by [u e_p]^T.

In [13]:
# and handle source term inclusion in the PDE system at each Runge-Kutta stage
def source_inclusion(rhs, u, k, src, src_index, src_deriv, e_p, t, dt, n_eq):
    """
    Add source terms to the PDE system at each Runge-Kutta stage.
    
    Parameters:
    -----------
    rhs : list
        Right-hand side of PDE system without sources
    u : list  
        Wavefield variables
    k : list
        Runge-Kutta stage variables
    src_index : list
        Source indices
    src_deriv : list
        Source derivatives
    e_p : list
        Expansion coefficients of the source term Taylor's series
    t : symbol
        Time symbol
    dt : symbol
        Time step symbol
    n_eq : int
        Number of equations
        
    Returns:
    --------
    src_lhs : list
        Operator application to the vector [u, e_p]^T including source terms
    e_p : list
        Updated expansion coefficients of the source term Taylor's series
    """
    # Replace wavefield variables with stage variables
    # src_rhs = [uxreplace(rhs[i], {u[m]: k[m] for m in range(n_eq)}) for i in range(n_eq)]
    src_rhs = rhs.subs({u[m]: k[m] for m in range(n_eq)})
    
    p = len(src_deriv)

    # Add source contributions
    src_vec = [0] * n_eq
    for i in range(p):
        if e_p[i] != 0:
            for j in range(len(src_index)):
                print("source index:", src_rhs[src_index[j]])
                src_vec[src_index[j]] += src[j][0] * src_deriv[i][j].subs({t: t * dt}) * e_p[i]
    src_rhs += VectorFunction(grid=u.grid, components=tuple(src_vec))
    
    # Update expansion coefficients of the source term Taylor's series
    e_p = [e_p[i] + dt * e_p[i + 1] for i in range(p - 1)] + [e_p[-1]]

    return src_rhs, e_p

print("  - source_inclusion(): Apply the spatial operator of the PDE system at RK stages")

  - source_inclusion(): Apply the spatial operator of the PDE system at RK stages


## 5. Implementation of the RK method

Construct the multi-stage time integration scheme with initialization, multiple RK stages, and final update. This implements a toy example of a class of High-Order Runge-Kutta (HORK) methods

\begin{align*}
        \boldsymbol{k}_0&=\boldsymbol{u}_n\\
        \boldsymbol{k}_1&=\left(\boldsymbol{I}_{n\times n}+\Delta t \hat{\boldsymbol{H}}\right)\boldsymbol{k}_{0},\\
        \boldsymbol{k}_2&=\left(\boldsymbol{I}_{n\times n}+\Delta t \hat{\boldsymbol{H}}\right)\boldsymbol{k}_{1},\\
        \boldsymbol{u}_{n+1}&=\sum\limits_{i=0}^{1}\alpha_i\boldsymbol{k}_i+\alpha_{2}\left(\boldsymbol{I}_{n\times n}+\Delta t \hat{\boldsymbol{H}}\right)\boldsymbol{k}_{2},        
\end{align*}

with proper source term integration.

In [14]:
n_eq = 5                  # Number of PDE unknowns (u, v)

# Temporary variables for Runge-Kutta stages
k = [TimeFunction(name=f'k{i}', grid=grid, space_order=2, time_order=0, dtype=u_multi_stage[i].dtype) for i in range(n_eq)]
k_vec = VectorTimeFunction(grid=grid, components=k)
# Previous stage variables needed for temporary storage
k0 = [TimeFunction(name=f'k0{i}', grid=grid, space_order=2, time_order=0, dtype=u_multi_stage[i].dtype) for i in range(n_eq)]
k_old = VectorTimeFunction(grid=grid, components=k0)
print(f"\nRunge-Kutta temporary variables:")
print(f"  k: {[ki.name for ki in k]}")
print(f"  k_old: {[ki.name for ki in k_old]}")

# Initialize expansion coefficients of the source term Taylor's series
e_p = [0] * deg
eta = 1
e_p[-1] = 1 / eta
print(f"\nExpansion coefficients initialized of the source term Taylor's series:")
print(f"  e_p = {e_p}")
print(f"  eta = {eta}")

# Initialize SSPRK coefficients
def ssprk_alpha(deg, mu=1.0):
        """
        Computes the coefficients for the Strong Stability Preserving Runge-Kutta (SSPRK) method.

        Parameters:
        mu : float
            Theoretically, it should be the inverse of the CFL condition (typically mu=1 for best performance).
            In practice, mu=1 works better.
        degree : int
            Degree of the polynomial used in the time-stepping scheme.

        Returns:
        numpy.ndarray
            Array of SSPRK coefficients.
        """

        alpha = [0] * deg
        alpha[0] = 1.0  # Initial coefficient

        # recurrence relation to compute the HORK coefficients following the formula in Gottlieb and Gottlieb (2002)
        for i in range(1, deg):
            alpha[i] = 1 / (mu * (i + 1)) * alpha[i - 1]
            alpha[1:i] = [1 / (mu * j) * alpha[j - 1] for j in range(1, i)]
            alpha[0] = 1 - sum(alpha[1:i + 1])

        return alpha

alpha = ssprk_alpha(deg)
print(f"\nSSPRK coefficients initialized:")
print(f"  alpha = {alpha}")


# Initialize list to store all stage equations
stage_eqs = []

print("Building Runge-Kutta-like scheme...")

# Stage 0: Initialization
print("  Stage 0: Initialization")
stage_eqs.extend([Eq(k_vec, u_multi_stage)])
stage_eqs.extend([Eq(u_multi_stage, u_multi_stage * alpha[0])])
# 
# stage_eqs.extend([Eq(k[i], U_multi_stage[i]) for i in range(n_eq)])
# [stage_eqs.append(Eq(U_multi_stage[i].forward, U_multi_stage[i] * alpha[0])) for i in range(n_eq)]

# Stage 1
print("  Stage 1: First RK stage")
stage_eqs.extend([Eq(k_old, k_vec)])
src_lhs, e_p = source_inclusion(system_eqs_rhs, u_multi_stage, k_old, src, src_index, src_deriv, e_p, t, dt, n_eq)
stage_eqs.extend([Eq(k_vec, k_old + dt * src_lhs)])
stage_eqs.extend([Eq(u_multi_stage, u_multi_stage + k_vec * alpha[1])])
# 
# [stage_eqs.append(Eq(k_old[j], k[j])) for j in range(n_eq)]
# src_lhs, e_p = source_inclusion(system_eqs_rhs, U_multi_stage, k_old, src_index, src_deriv, e_p, t, dt, n_eq)
# [stage_eqs.append(Eq(k[j], k_old[j] + dt * src_lhs[j])) for j in range(n_eq)]
# [stage_eqs.append(Eq(U_multi_stage[j].forward, U_multi_stage[j].forward + k[j] * alpha[1])) for j in range(n_eq)]

# Stage 2
print("  Stage 2: Second RK stage")
stage_eqs.extend([Eq(k_old, k_vec)])
src_lhs, e_p = source_inclusion(system_eqs_rhs, u_multi_stage, k_old, src, src_index, src_deriv, e_p, t, dt, n_eq)
stage_eqs.extend([Eq(k_vec, k_old + dt * src_lhs)])
# 
# [stage_eqs.append(Eq(k_old[j], k[j])) for j in range(n_eq)]
# src_lhs, e_p = source_inclusion(system_eqs_rhs, U_multi_stage, k_old, src_index, src_deriv, e_p, t, dt, n_eq)
# [stage_eqs.append(Eq(k[j], k_old[j] + dt * src_lhs[j])) for j in range(n_eq)]

# Stage 3 and final update
print("  Stage 3: Final RK stage and update")
stage_eqs.extend([Eq(k_old, k_vec)])
src_lhs, _ = source_inclusion(system_eqs_rhs, u_multi_stage, k_old, src, src_index, src_deriv, e_p, t, dt, n_eq)
stage_eqs.extend([Eq(k_vec, k_old + dt * src_lhs)])
stage_eqs.extend([Eq(u_multi_stage, u_multi_stage + k_vec * alpha[deg-1])])
# 
# [stage_eqs.append(Eq(k_old[j], k[j])) for j in range(n_eq)]
# src_lhs, _ = source_inclusion(system_eqs_rhs, U_multi_stage, k_old, src_index, src_deriv, e_p, t, dt, n_eq)
# [stage_eqs.append(Eq(k[j], k_old[j] + dt * src_lhs[j])) for j in range(n_eq)]
# [stage_eqs.append(Eq(U_multi_stage[j].forward, U_multi_stage[j].forward + k[j] * alpha[deg - 1])) for j in range(n_eq)]

print(f"\nTotal equations created: {len(stage_eqs)}")
print("Scheme structure:")
for i, stage in enumerate(stage_eqs):
    print(f"  Stage {i}: {stage}")



Runge-Kutta temporary variables:
  k: ['k0', 'k1', 'k2', 'k3', 'k4']
  k_old: ['k00', 'k01', 'k02', 'k03', 'k04']

Expansion coefficients initialized of the source term Taylor's series:
  e_p = [0, 0, 1.0]
  eta = 1

SSPRK coefficients initialized:
  alpha = [0.33333333333333337, 0.5, 0.16666666666666666]
Building Runge-Kutta-like scheme...
  Stage 0: Initialization
  Stage 1: First RK stage
source index: Derivative(k02(t, x, y), x) + Derivative(k03(t, x, y), y)
source index: Derivative(k02(t, x, y), x) + Derivative(k03(t, x, y), y)
source index: Derivative(k03(t, x, y), x) + Derivative(k04(t, x, y), y)
  Stage 2: Second RK stage
source index: Derivative(k02(t, x, y), x) + Derivative(k03(t, x, y), y)
source index: Derivative(k02(t, x, y), x) + Derivative(k03(t, x, y), y)
source index: Derivative(k03(t, x, y), x) + Derivative(k04(t, x, y), y)
source index: Derivative(k02(t, x, y), x) + Derivative(k03(t, x, y), y)
source index: Derivative(k02(t, x, y), x) + Derivative(k03(t, x, y), y)
s

## 6. Create and Run Devito Operator

Compile all the equations into a Devito Operator and execute the simulation with the specified parameters.

In [15]:
# Create the Devito Operator
print("Creating Devito Operator...")
op = Operator(stage_eqs, subs=grid.spacing_map)
print("\nDevito Operator created.")
print(op)

print("Operator successfully created!")
print(f"  Number of equations: {len(stage_eqs)}")
print(f"  Grid spacing substitutions applied: {grid.spacing_map}")

# Define simulation parameters
dt_value = 0.001    # Time step size
time_max = 2000     # Maximum simulation time

print(f"\nSimulation parameters:")
print(f"  Time step (dt): {dt_value}")
print(f"  Maximum time: {time_max}")

# Execute the simulation
print("\nRunning simulation...")
op(dt=dt_value, time=time_max)

print("Simulation completed successfully!")

# Display final wavefield shapes and some statistics
print(f"\nFinal wavefield statistics:")
for i, u in enumerate(u_multi_stage):
    print(f"  {fun_labels[i]}:")
    print(f"    Shape: {u.data.shape}")
    print(f"    Max value: {np.max(u.data):.6f}")
    print(f"    Min value: {np.min(u.data):.6f}")
    print(f"    Mean value: {np.mean(u.data):.6f}")

Creating Devito Operator...


Operator `Kernel` generated in 1.66 s
  * lowering.Clusters: 1.29 s (78.1 %)
     * schedule: 0.74 s (44.8 %)
     * specializing.Clusters: 0.45 s (27.3 %)
Flops reduction after symbolic optimization: [320 --> 174]



Devito Operator created.
/* Devito generated code for Operator `Kernel` */

#define _POSIX_C_SOURCE 200809L
#define START(S) struct timeval start_ ## S , end_ ## S ; gettimeofday(&start_ ## S , NULL);
#define STOP(S,T) gettimeofday(&end_ ## S, NULL); T->S += (double)(end_ ## S .tv_sec-start_ ## S.tv_sec)+(double)(end_ ## S .tv_usec-start_ ## S .tv_usec)/1000000;

#include "stdlib.h"
#include "math.h"
#include "sys/time.h"
#include "xmmintrin.h"
#include "pmmintrin.h"

struct dataobj
{
  void *restrict data;
  int * size;
  unsigned long nbytes;
  unsigned long * npsize;
  unsigned long * dsize;
  int * hsize;
  int * hofs;
  int * oofs;
  void * dmap;
} ;

struct profiler
{
  double section0;
  double section1;
  double section2;
  double section3;
  double section4;
} ;


int Kernel(struct dataobj *restrict k0_vec, struct dataobj *restrict k00_vec, struct dataobj *restrict k01_vec, struct dataobj *restrict k02_vec, struct dataobj *restrict k03_vec, struct dataobj *restrict k04_vec, s

Operator `Kernel` jit-compiled `/tmp/devito-jitcache-uid1000/d75ed6c09ab65d33734dd7c52bd07f4f9be6adbd.c` in 0.33 s with `CustomCompiler`
Allocating host memory for k0(1, 205, 205) [328 KB]
Allocating host memory for k00(1, 205, 205) [328 KB]
Allocating host memory for k01(1, 205, 205) [328 KB]
Allocating host memory for k02(1, 205, 205) [328 KB]
Allocating host memory for k03(1, 205, 205) [328 KB]
Allocating host memory for k04(1, 205, 205) [328 KB]
Allocating host memory for k1(1, 205, 205) [328 KB]
Allocating host memory for k2(1, 205, 205) [328 KB]
Allocating host memory for k3(1, 205, 205) [328 KB]
Allocating host memory for k4(1, 205, 205) [328 KB]
Allocating host memory for txx(1, 205, 205) [328 KB]
Allocating host memory for txy(1, 205, 205) [328 KB]
Allocating host memory for tyy(1, 205, 205) [328 KB]
Allocating host memory for vx(1, 205, 205) [328 KB]
Allocating host memory for vy(1, 205, 205) [328 KB]
Operator `Kernel` arguments preprocess: 0.03 s
Operator `Kernel` arguments 

Simulation completed successfully!

Final wavefield statistics:
  vx:
    Shape: (1, 201, 201)
    Max value: inf
    Min value: -inf
    Mean value: nan
  vy:
    Shape: (1, 201, 201)
    Max value: inf
    Min value: -inf
    Mean value: nan
  txx:
    Shape: (1, 201, 201)
    Max value: inf
    Min value: -inf
    Mean value: nan
  txy:
    Shape: (1, 201, 201)
    Max value: inf
    Min value: -inf
    Mean value: nan
  tyy:
    Shape: (1, 201, 201)
    Max value: inf
    Min value: -inf
    Mean value: nan


/home/fernan/anaconda3/envs/devito_b/lib/python3.12/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


## 7. References


<a id="ref-higham2010"></a>
[1] Al-Mohy AH, Higham NJ (2010) A new scaling and squaring algorithm for
the matrix exponential. SIAM Journal on Matrix Analysis and Applications
31(3):970–989

<a id="ref-gottlieb2003"></a>
[2] Gottlieb S, Gottlieb LAJ (2003) Strong stability preserving properties of runge–kutta time discretization methods for linear constant coefficient operators. Journal of Scientific Computing 18(1):83–109